In [11]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv
import os

In [12]:
load_dotenv()

openai_ef = OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="text-embedding-3-small"
)

In [13]:
chroma_client = chromadb.Client()

# chroma_client.delete_collection(name="CVs")

collection = chroma_client.get_or_create_collection(
    name="CVs",
    embedding_function=openai_ef
)

In [14]:
documents = [
    "Frontend developer specialized in React and UI design",
    
    "Backend developer specialized in Python APIs and databases",
    
    "Cybersecurity specialist certified CEH with Linux experience",
    
    "Digital marketing expert focused on product promotion and social media",
    
    "AI engineer specialized in NLP and Machine Learning"
]

In [15]:
metadatas = [
    {
        "name": "Giulia Rossi",
        "role": "Frontend Developer",
        "info": "Expert in React, JavaScript, HTML and CSS"
    },

    {
        "name": "Marco Bianchi",
        "role": "Backend Developer",
        "info": "Expert in Python, APIs and PostgreSQL"
    },

    {
        "name": "Luca Verdi",
        "role": "Cybersecurity Specialist",
        "info": "CEH certified ethical hacker with Linux skills"
    },

    {
        "name": "Sara Neri",
        "role": "Digital Marketing Specialist",
        "info": "Expert in advertising campaigns and product promotion"
    },

    {
        "name": "Andrea Gallo",
        "role": "AI Engineer",
        "info": "Specialized in NLP and Deep Learning"
    }
]

In [16]:
ids = ["1", "2", "3", "4", "5"]

In [17]:
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

In [18]:
collection.get()

{'ids': ['1', '2', '3', '4', '5'],
 'embeddings': None,
 'documents': ['Frontend developer specialized in React and UI design',
  'Backend developer specialized in Python APIs and databases',
  'Cybersecurity specialist certified CEH with Linux experience',
  'Digital marketing expert focused on product promotion and social media',
  'AI engineer specialized in NLP and Machine Learning'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'name': 'Giulia Rossi',
   'info': 'Expert in React, JavaScript, HTML and CSS',
   'role': 'Frontend Developer'},
  {'name': 'Marco Bianchi',
   'role': 'Backend Developer',
   'info': 'Expert in Python, APIs and PostgreSQL'},
  {'name': 'Luca Verdi',
   'info': 'CEH certified ethical hacker with Linux skills',
   'role': 'Cybersecurity Specialist'},
  {'role': 'Digital Marketing Specialist',
   'name': 'Sara Neri',
   'info': 'Expert in advertising campaigns and product promotion'},
  {'role': 'AI Engineer',
   'na

In [19]:
user_question = "mi serve qualcuno per promuovere il mio prodotto"

# user_question = "mi serve qualcuno per mettere in sicurezza il mio sito web"

# user_question = "mi serve qualcuno certificato CEH"

In [20]:
results = collection.query(
    query_texts=[user_question],
    n_results=1
)

In [21]:
results

{'ids': [['4']],
 'embeddings': None,
 'documents': [['Digital marketing expert focused on product promotion and social media']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'role': 'Digital Marketing Specialist',
    'name': 'Sara Neri',
    'info': 'Expert in advertising campaigns and product promotion'}]],
 'distances': [[0.5171064138412476]]}

In [22]:
results['metadatas'][0][0]['info']

'Expert in advertising campaigns and product promotion'

In [23]:
results = collection.query(
    query_texts=[user_question],
    n_results=2
)

In [24]:
retrieved_docs = results['documents'][0]
retrieved_metadata = results['metadatas'][0]

In [25]:
context = ""

for i in range(len(retrieved_docs)):

    context += f"""
Candidate:
{retrieved_metadata[i]['name']}

Role:
{retrieved_metadata[i]['role']}

Info:
{retrieved_metadata[i]['info']}

CV:
{retrieved_docs[i]}

-------------------
"""

In [26]:
prompt = f"""
You are an AI recruiter assistant.

Use the following candidates information to answer the user's request.

Candidates:
{context}

User Request:
{user_question}

Provide:
- the best candidate
- why this candidate matches the request
- a short professional explanation
"""

In [28]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "Tu sei un AI recruiter e traducimi tutto in italiano."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

print(completion.choices[0].message.content)

Candidata migliore:
Sara Neri

Motivo per cui questa candidata soddisfa la richiesta:
Sara Neri è un'esperta in campagne pubblicitarie e promozione di prodotti, il che la rende particolarmente adatta per aiutarti a promuovere il tuo prodotto.

Spiegazione professionale:
Sara ha una solida esperienza nel marketing digitale, con un focus specifico sulla promozione di prodotti e sull'utilizzo dei social media. Le sue competenze nella gestione delle campagne pubblicitarie le permetteranno di creare strategie efficaci per aumentare la visibilità e le vendite del tuo prodotto.
